# U1 — Quantitative Metrics (RQ3)

Loads per-sample results produced by `notebooks/_run_u1.py` and aggregates
them into `summary.{csv,md,tex}` with bootstrap 95% CIs.

**Heavy compute lives in `_run_u1.py`** (model loading, generation, decoding).
This notebook only loads its outputs, filters to the 3 directions of interest,
and writes the summary tables.

**Directions kept:**

| Direction | Metric |
|---|---|
| `tok_rgb@256_within` | Teacher-forced perplexity (RGB within-modality, 25%→75%) |
| `rgb_caption` | CLIPScore + BLEU-4 |
| `caption_rgb` | Pixel-MSE + SSIM |

All other directions present in `per_sample/*.json` (leftovers from older runs)
are filtered out.


In [26]:
import csv
import json
from collections import defaultdict
from pathlib import Path

import numpy as np

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
METRICS_DIR = PROJECT_DIR / 'results' / 'milestone2' / 'metrics'
PER_SAMPLE_DIR = METRICS_DIR / 'per_sample'

# The 3 directions produced by _run_u1.py
ALLOWED_DIRECTIONS = {'tok_rgb@256_within', 'rgb_caption', 'caption_rgb'}

# Checkpoints to include (matches _run_u1.py)
CHECKPOINTS = ['mor_5000_3r', 'mor_5000_4r', 'baseline_vanilla', 'random_router_5000']

N_BOOTSTRAP = 1000
RNG_SEED = 42

print(f'PROJECT_DIR     = {PROJECT_DIR}')
print(f'PER_SAMPLE_DIR  = {PER_SAMPLE_DIR}')
print(f'Allowed dirs    = {sorted(ALLOWED_DIRECTIONS)}')


PROJECT_DIR     = /home/hugues/Documents/Visual_Intelligence/Mixture-of-Recursions-for-Vision
PER_SAMPLE_DIR  = /home/hugues/Documents/Visual_Intelligence/Mixture-of-Recursions-for-Vision/results/milestone2/metrics/per_sample
Allowed dirs    = ['caption_rgb', 'rgb_caption', 'tok_rgb@256_within']


In [27]:
def bootstrap_ci(arr: np.ndarray, n: int = N_BOOTSTRAP) -> dict:
    if arr.size == 0:
        return {'mean': np.nan, 'lo': np.nan, 'hi': np.nan}
    rng = np.random.default_rng(RNG_SEED)
    idx = rng.integers(0, len(arr), (n, len(arr)))
    boot = arr[idx].mean(axis=1)
    return {'mean': float(arr.mean()),
            'lo': float(np.percentile(boot, 2.5)),
            'hi': float(np.percentile(boot, 97.5))}


# FLOPs accounting — read U2's per-modality mean depth if available
U2_JSON_PATH = PROJECT_DIR / 'results' / 'milestone2' / 'router' / 'per_modality_depth.json'
_U2_DATA = json.load(open(U2_JSON_PATH)) if U2_JSON_PATH.exists() else {}

CKPT_NR = {'mor_5000_3r': 3, 'mor_5000_4r': 4, 'baseline_vanilla': None, 'random_router_5000': 3}

def load_u2_mean_depth(label: str, modality: str) -> float | None:
    return _U2_DATA.get(label, {}).get('per_modality_stats', {}).get(modality, {}).get('mean')

def effective_flops_ratio(label: str, modality: str) -> float:
    nr = CKPT_NR.get(label)
    mean_depth = load_u2_mean_depth(label, modality)
    if nr is None or mean_depth is None:
        return 1.0
    return mean_depth / nr

print('Helpers ready.')


Helpers ready.


## Load per-sample results

Reads `results/milestone2/metrics/per_sample/*.json` (written by `_run_u1.py`),
drops any rows whose `direction` is not in `ALLOWED_DIRECTIONS`, and tags each
row with an `effective_flops_ratio` derived from U2's per-modality mean depth.


In [28]:
all_rows = []
file_count = defaultdict(int)
direction_counts = defaultdict(int)
dropped_directions = defaultdict(int)

for ckpt in CHECKPOINTS:
    files = sorted(PER_SAMPLE_DIR.glob(f'{ckpt}_*.json'))
    file_count[ckpt] = len(files)
    for fp in files:
        try:
            rows = json.load(open(fp))
        except Exception as e:
            print(f'  skip {fp.name}: {e}')
            continue
        for r in rows:
            d = r.get('direction')
            if d not in ALLOWED_DIRECTIONS:
                dropped_directions[d] += 1
                continue
            # Use modality that drives FLOPs: same as direction's source/target group
            mod_for_flops = 'tok_rgb@256' if 'rgb' in d else 'caption'
            r['flops_ratio'] = effective_flops_ratio(r['checkpoint'], mod_for_flops)
            r['mean_depth'] = load_u2_mean_depth(r['checkpoint'], mod_for_flops)
            all_rows.append(r)
            direction_counts[(r['checkpoint'], d)] += 1

print(f'Loaded {len(all_rows)} rows from {sum(file_count.values())} per_sample files.')
print()
print('Per-checkpoint file counts:')
for c in CHECKPOINTS:
    print(f'  {c:25s} {file_count[c]} files')
print()
print('Per-(checkpoint,direction) kept-row counts:')
for (c, d), n in sorted(direction_counts.items()):
    print(f'  {c:25s} {d:25s} {n}')
print()
if dropped_directions:
    print('Dropped directions (not in ALLOWED_DIRECTIONS):')
    for d, n in sorted(dropped_directions.items()):
        print(f'  {d:25s} {n}')


Loaded 7794 rows from 2598 per_sample files.

Per-checkpoint file counts:
  mor_5000_3r               699 files
  mor_5000_4r               699 files
  baseline_vanilla          699 files
  random_router_5000        501 files

Per-(checkpoint,direction) kept-row counts:
  baseline_vanilla          caption_rgb               699
  baseline_vanilla          rgb_caption               699
  baseline_vanilla          tok_rgb@256_within        699
  mor_5000_3r               caption_rgb               699
  mor_5000_3r               rgb_caption               699
  mor_5000_3r               tok_rgb@256_within        699
  mor_5000_4r               caption_rgb               699
  mor_5000_4r               rgb_caption               699
  mor_5000_4r               tok_rgb@256_within        699
  random_router_5000        caption_rgb               501
  random_router_5000        rgb_caption               501
  random_router_5000        tok_rgb@256_within        501

Dropped directions (not in ALLOW

## Aggregate with bootstrap 95% CIs


In [29]:
METRIC_FIELDS = ['tf_perplexity', 'pixel_mse', 'ssim', 'clip_score', 'bleu4']

agg = defaultdict(lambda: defaultdict(list))     # (ckpt, dir) -> field -> [values]
failures = defaultdict(lambda: {'total': 0, 'failed': 0})

for row in all_rows:
    key = (row['checkpoint'], row['direction'])
    failures[key]['total'] += 1
    if row.get('generation_failure'):
        failures[key]['failed'] += 1
    else:
        for f in METRIC_FIELDS:
            v = row.get(f)
            if v is not None and not (isinstance(v, float) and np.isnan(v)):
                agg[key][f].append(v)

summary_rows = []
for (ckpt, direction), metrics in agg.items():
    row = {'checkpoint': ckpt, 'direction': direction}
    total = failures[(ckpt, direction)]['total']
    failed = failures[(ckpt, direction)]['failed']
    row['n_samples'] = total
    row['failure_rate'] = round(failed / total, 3) if total > 0 else 0.0

    for f in METRIC_FIELDS:
        vals = np.array(metrics.get(f, []), dtype=float)
        if vals.size == 0:
            row[f] = None
            row[f'{f}_ci_lo'] = None
            row[f'{f}_ci_hi'] = None
        else:
            ci = bootstrap_ci(vals)
            row[f] = round(ci['mean'], 4)
            row[f'{f}_ci_lo'] = round(ci['lo'], 4)
            row[f'{f}_ci_hi'] = round(ci['hi'], 4)

    flops_vals = [r['flops_ratio'] for r in all_rows
                  if r['checkpoint'] == ckpt and r['direction'] == direction
                  and r.get('flops_ratio') is not None]
    row['effective_flops_ratio'] = round(float(np.mean(flops_vals)), 4) if flops_vals else None
    summary_rows.append(row)

# Sort for deterministic output: checkpoint, then by direction in a stable order
DIRECTION_ORDER = {'tok_rgb@256_within': 0, 'rgb_caption': 1, 'caption_rgb': 2}
summary_rows.sort(key=lambda r: (r['checkpoint'], DIRECTION_ORDER.get(r['direction'], 99)))
print(f'Summary rows: {len(summary_rows)}')
for r in summary_rows:
    print(f"  {r['checkpoint']:25s} {r['direction']:25s} "
          f"n={r['n_samples']:>4d} fail={r['failure_rate']:.3f} "
          f"ppl={r.get('tf_perplexity')} mse={r.get('pixel_mse')} "
          f"clip={r.get('clip_score')} bleu={r.get('bleu4')}")


Summary rows: 12
  baseline_vanilla          tok_rgb@256_within        n= 699 fail=0.000 ppl=56.5627 mse=None clip=None bleu=None
  baseline_vanilla          rgb_caption               n= 699 fail=0.011 ppl=None mse=None clip=34.1216 bleu=0.4217
  baseline_vanilla          caption_rgb               n= 699 fail=0.000 ppl=None mse=695.7513 clip=None bleu=None
  mor_5000_3r               tok_rgb@256_within        n= 699 fail=0.000 ppl=73.0771 mse=None clip=None bleu=None
  mor_5000_3r               rgb_caption               n= 699 fail=0.001 ppl=None mse=None clip=33.9183 bleu=0.3941
  mor_5000_3r               caption_rgb               n= 699 fail=0.000 ppl=None mse=658.4759 clip=None bleu=None
  mor_5000_4r               tok_rgb@256_within        n= 699 fail=0.000 ppl=84.0775 mse=None clip=None bleu=None
  mor_5000_4r               rgb_caption               n= 699 fail=0.003 ppl=None mse=None clip=34.0684 bleu=0.3942
  mor_5000_4r               caption_rgb               n= 699 fail=0.006

## Write `summary.{csv,md,tex}`


In [30]:
# CSV
all_keys = ['checkpoint', 'direction', 'n_samples', 'failure_rate', 'effective_flops_ratio']
for f in METRIC_FIELDS:
    all_keys += [f, f'{f}_ci_lo', f'{f}_ci_hi']

csv_path = METRICS_DIR / 'summary.csv'
with open(csv_path, 'w', newline='') as fp:
    w = csv.DictWriter(fp, fieldnames=all_keys, extrasaction='ignore')
    w.writeheader()
    for row in summary_rows:
        w.writerow(row)
print(f'Saved {csv_path}')

# Markdown
PRIMARY = {'tok_rgb@256_within': 'tf_perplexity',
           'rgb_caption': 'clip_score',
           'caption_rgb': 'pixel_mse'}

md_path = METRICS_DIR / 'summary.md'
with open(md_path, 'w') as f:
    f.write('| Checkpoint | Direction | Primary metric | 95% CI | FLOPs ratio | Failure rate | N |\n')
    f.write('|---|---|---|---|---|---|---|\n')
    for row in summary_rows:
        primary = PRIMARY.get(row['direction'], 'tf_perplexity')
        val = row.get(primary); lo = row.get(f'{primary}_ci_lo'); hi = row.get(f'{primary}_ci_hi')
        val_str = f'{primary}={val:.4f} [{lo:.4f}, {hi:.4f}]' if val is not None else f'{primary}=n/a'
        fr = row.get('effective_flops_ratio')
        fr_str = f'{fr:.3f}' if fr is not None else 'n/a'
        f.write(f'| {row["checkpoint"]} | {row["direction"]} | {val_str} | '
                f'{fr_str} | {row["failure_rate"]:.3f} | {row["n_samples"]} |\n')
print(f'Saved {md_path}')

# LaTeX (booktabs)
tex_path = METRICS_DIR / 'summary.tex'
with open(tex_path, 'w') as f:
    f.write('% Generated by 2026-05-19_u1_quantitative_metrics.ipynb\n')
    f.write('% Add \\caption{} and \\label{} before including in report\n\n')
    f.write('\\begin{table}[ht]\n\\centering\n')
    f.write('\\begin{tabular}{llrrrrr}\n\\toprule\n')
    f.write('Checkpoint & Direction & Primary metric & 95\\% CI & FLOPs ratio & Fail rate & N \\\\\n')
    f.write('\\midrule\n')
    for row in summary_rows:
        primary = PRIMARY.get(row['direction'], 'tf_perplexity')
        val = row.get(primary); lo = row.get(f'{primary}_ci_lo'); hi = row.get(f'{primary}_ci_hi')
        val_str = f'{val:.4f}' if val is not None else 'n/a'
        ci_str = f'[{lo:.4f}, {hi:.4f}]' if lo is not None else '---'
        fr = row.get('effective_flops_ratio')
        fr_str = f'{fr:.3f}' if fr is not None else 'n/a'
        f.write(f'{row["checkpoint"].replace("_","-")} & {row["direction"].replace("_","-")} & '
                f'{val_str} & {ci_str} & {fr_str} & {row["failure_rate"]:.3f} & {row["n_samples"]} \\\\\n')
    f.write('\\bottomrule\n\\end{tabular}\n\\end{table}\n')
print(f'Saved {tex_path}')


Saved /home/hugues/Documents/Visual_Intelligence/Mixture-of-Recursions-for-Vision/results/milestone2/metrics/summary.csv
Saved /home/hugues/Documents/Visual_Intelligence/Mixture-of-Recursions-for-Vision/results/milestone2/metrics/summary.md
Saved /home/hugues/Documents/Visual_Intelligence/Mixture-of-Recursions-for-Vision/results/milestone2/metrics/summary.tex


## Failure-rate report
Flag any (checkpoint, direction) with failure rate > 20%.


In [31]:
print('=== Failure rate report ===')
for row in summary_rows:
    fr = row['failure_rate']
    flag = ' <-- FLAG: >20% failure rate' if fr > 0.2 else ''
    print(f'  {row["checkpoint"]:25s} {row["direction"]:25s} n={row["n_samples"]:>4d} fail={fr:.3f}{flag}')


=== Failure rate report ===
  baseline_vanilla          tok_rgb@256_within        n= 699 fail=0.000
  baseline_vanilla          rgb_caption               n= 699 fail=0.011
  baseline_vanilla          caption_rgb               n= 699 fail=0.000
  mor_5000_3r               tok_rgb@256_within        n= 699 fail=0.000
  mor_5000_3r               rgb_caption               n= 699 fail=0.001
  mor_5000_3r               caption_rgb               n= 699 fail=0.000
  mor_5000_4r               tok_rgb@256_within        n= 699 fail=0.000
  mor_5000_4r               rgb_caption               n= 699 fail=0.003
  mor_5000_4r               caption_rgb               n= 699 fail=0.006
  random_router_5000        tok_rgb@256_within        n= 501 fail=0.000
  random_router_5000        rgb_caption               n= 501 fail=0.038
  random_router_5000        caption_rgb               n= 501 fail=0.058


## Pivot table (limited to first 500 samples, matching `_run_u1.py`)

Restricts to the first 500 sorted sample IDs (the same set `_run_u1.py` iterates
over) and rebuilds the summary as a single pivoted table:

- **Rows** = direction (3)
- **Super-columns** = model (4)
- **Sub-columns** = the metrics relevant to each direction (filled per row)

Each cell shows `mean [lo, hi]` from a 1k-resample bootstrap 95% CI.
Saved to `summary_pivot.{md,tex,csv}`.


In [ ]:
# ── 1. Restrict to first 500 sample IDs at aug_idx=0 (matches _run_u1.py) ──
N_SAMPLES_CAP = 500
all_sids = sorted({r['sid'] for r in all_rows if r.get('aug_idx', 0) == 0})
SID_KEEP = set(all_sids[:N_SAMPLES_CAP])
rows500 = [r for r in all_rows
           if r['sid'] in SID_KEEP and r.get('aug_idx', 0) == 0]
print(f'Capped to {len(SID_KEEP)} sids (aug_idx=0); '
      f'rows kept: {len(rows500)} (was {len(all_rows)})')

# ── 2. Re-aggregate on the capped set ──
agg500 = defaultdict(lambda: defaultdict(list))
fail500 = defaultdict(lambda: {'total': 0, 'failed': 0})
for r in rows500:
    k = (r['checkpoint'], r['direction'])
    fail500[k]['total'] += 1
    if r.get('generation_failure'):
        fail500[k]['failed'] += 1
    else:
        for f in METRIC_FIELDS:
            v = r.get(f)
            if v is not None and not (isinstance(v, float) and np.isnan(v)):
                agg500[k][f].append(v)

def _mean_std(vals):
    arr = np.array(vals, dtype=float)
    if arr.size == 0:
        return None, None
    return float(arr.mean()), float(arr.std(ddof=1)) if arr.size > 1 else 0.0

# ── 3. Metric definitions: field, label, lower-is-better? ──
DIRECTION_METRICS = {
    'tok_rgb@256_within': [('tf_perplexity', 'PPL ↓', True)],
    'rgb_caption':        [('clip_score', 'CLIPScore ↑', False),
                           ('bleu4', 'BLEU-4 ↑', False)],
    'caption_rgb':        [('pixel_mse', 'Pixel-MSE ↓', True),
                           ('ssim', 'SSIM ↑', False)],
}
DIRECTION_LABELS = {
    'tok_rgb@256_within': 'RGB within (25%→75%)',
    'rgb_caption':        'RGB → Caption',
    'caption_rgb':        'Caption → RGB',
}
# Order: baselines (Vanilla, Random) first, then MoR variants
MODELS = ['baseline_vanilla', 'random_router_5000', 'mor_5000_3r', 'mor_5000_4r']
MODEL_LABELS = {
    'baseline_vanilla':   'Vanilla',
    'random_router_5000': 'Random Nr=3',
    'mor_5000_3r':        'MoR Nr=3',
    'mor_5000_4r':        'MoR Nr=4',
}
ORDERED_DIRS = ['tok_rgb@256_within', 'rgb_caption', 'caption_rgb']

# ── 4. Build the pivot, recording which cell is best per (direction, metric) ──
pivot = {}  # direction -> model -> [(metric_label, "mean ± std", is_best), ...]
for d in ORDERED_DIRS:
    pivot[d] = {m: [] for m in MODELS}
    for field, label, lower_better in DIRECTION_METRICS[d]:
        means = {}
        stats = {}
        for m in MODELS:
            mean, std = _mean_std(agg500[(m, d)].get(field, []))
            stats[m] = (mean, std)
            if mean is not None:
                means[m] = mean
        # Find best model for this (direction, metric)
        if means:
            best_model = min(means, key=means.get) if lower_better else max(means, key=means.get)
        else:
            best_model = None
        for m in MODELS:
            mean, std = stats[m]
            if mean is None:
                txt = '—'
            else:
                txt = f'{mean:.3f} ± {std:.3f}'
            pivot[d][m].append((label, txt, m == best_model))

# ── 5. Console pretty-print (best in UPPERCASE-bold via ** markers) ──
print()
print(f'=== Pivot table (N≤{N_SAMPLES_CAP} samples per cell, mean ± std; best in **bold**) ===')
COLW = 26
for d in ORDERED_DIRS:
    print(f'\n{DIRECTION_LABELS[d]}  ({d})')
    metric_labels = [lbl for lbl, _, _ in pivot[d][MODELS[0]]]
    print('  ' + 'Metric'.ljust(14) + ''.join(MODEL_LABELS[m].rjust(COLW) for m in MODELS))
    for i, mlbl in enumerate(metric_labels):
        cells = []
        for m in MODELS:
            _, txt, is_best = pivot[d][m][i]
            cells.append((f'**{txt}**' if is_best else txt).rjust(COLW))
        print('  ' + mlbl.ljust(14) + ''.join(cells))
    fail_cells = []
    for m in MODELS:
        f = fail500[(m, d)]
        fail_cells.append(f'N={f["total"]} fail={f["failed"] / max(f["total"], 1):.3f}')
    print('  ' + ''.ljust(14) + ''.join(c.rjust(COLW) for c in fail_cells))


In [ ]:
# ── Write pivot summary files (mean ± std, best in bold) ──

def _bold_md(txt: str, is_best: bool) -> str:
    return f'**{txt}**' if is_best and txt != '—' else txt

def _bold_tex(txt: str, is_best: bool) -> str:
    s = txt.replace('±', '$\\pm$')
    return f'\\textbf{{{s}}}' if is_best and txt != '—' else s

# Markdown
md_lines = []
md_lines.append('| Direction | Metric | ' + ' | '.join(MODEL_LABELS[m] for m in MODELS) + ' |')
md_lines.append('|---|---|' + '---|' * len(MODELS))
for d in ORDERED_DIRS:
    metric_labels = [lbl for lbl, _, _ in pivot[d][MODELS[0]]]
    for i, mlbl in enumerate(metric_labels):
        dir_cell = DIRECTION_LABELS[d] if i == 0 else ''
        cells = [_bold_md(pivot[d][m][i][1], pivot[d][m][i][2]) for m in MODELS]
        md_lines.append(f'| {dir_cell} | {mlbl} | ' + ' | '.join(cells) + ' |')
pivot_md = METRICS_DIR / 'summary_pivot.md'
pivot_md.write_text('\n'.join(md_lines) + '\n')
print(f'Saved {pivot_md}')

# LaTeX (one tabular per direction)
tex_lines = ['% Generated by 2026-05-19_u1_quantitative_metrics.ipynb',
             f'% Pivot tables — N≤{N_SAMPLES_CAP} samples; mean $\\pm$ std; best per row in \\textbf{{bold}}\n']
for d in ORDERED_DIRS:
    metric_labels = [lbl for lbl, _, _ in pivot[d][MODELS[0]]]
    ncol_per_model = len(metric_labels)
    col_spec = 'l' + 'c' * (len(MODELS) * ncol_per_model)
    tex_lines.append('\\begin{table}[ht]\\centering')
    tex_lines.append(f'\\caption{{{DIRECTION_LABELS[d]} — mean $\\pm$ std}}')
    tex_lines.append(f'\\begin{{tabular}}{{{col_spec}}}')
    tex_lines.append('\\toprule')
    header = ['']
    for m in MODELS:
        header.append(f'\\multicolumn{{{ncol_per_model}}}{{c}}{{{MODEL_LABELS[m]}}}')
    tex_lines.append(' & '.join(header) + ' \\\\')
    sub = ['']
    for _m in MODELS:
        for mlbl in metric_labels:
            sub.append(mlbl.replace('↓', '$\\downarrow$').replace('↑', '$\\uparrow$'))
    tex_lines.append(' & '.join(sub) + ' \\\\')
    tex_lines.append('\\midrule')
    row = [DIRECTION_LABELS[d]]
    for m in MODELS:
        for i in range(ncol_per_model):
            _, txt, is_best = pivot[d][m][i]
            row.append(_bold_tex(txt, is_best))
    tex_lines.append(' & '.join(row) + ' \\\\')
    tex_lines.append('\\bottomrule')
    tex_lines.append('\\end{tabular}\\end{table}\n')
pivot_tex = METRICS_DIR / 'summary_pivot.tex'
pivot_tex.write_text('\n'.join(tex_lines))
print(f'Saved {pivot_tex}')

# CSV (long format with mean, std, is_best flag)
csv_path = METRICS_DIR / 'summary_pivot.csv'
with open(csv_path, 'w', newline='') as fp:
    w = csv.writer(fp)
    w.writerow(['direction', 'metric', 'model', 'mean', 'std',
                'is_best', 'n_samples', 'failure_rate'])
    for d in ORDERED_DIRS:
        for mi, (field, label, _lb) in enumerate(DIRECTION_METRICS[d]):
            for m in MODELS:
                vals = np.array(agg500[(m, d)].get(field, []), dtype=float)
                f = fail500[(m, d)]
                fr = f['failed'] / max(f['total'], 1)
                is_best = pivot[d][m][mi][2]
                if vals.size == 0:
                    w.writerow([d, field, m, '', '', int(is_best), f['total'], f'{fr:.4f}'])
                else:
                    std = vals.std(ddof=1) if vals.size > 1 else 0.0
                    w.writerow([d, field, m,
                                f'{vals.mean():.4f}', f'{std:.4f}',
                                int(is_best), f['total'], f'{fr:.4f}'])
print(f'Saved {csv_path}')

# Render inline
try:
    from IPython.display import Markdown, display
    display(Markdown('\n'.join(md_lines)))
except ImportError:
    print()
    print('\n'.join(md_lines))
